In [79]:
import polars as pl
from pathlib import Path

GTFS_DIRS = [
    "raw/gtfs/manhattan",
    "raw/gtfs/gtfs_m_may_june",
    "raw/gtfs/gtfs_m_march",
    "raw/gtfs/gtfs_m_april",
]

def load_gtfs(filename):
    dfs = []

    for folder in GTFS_DIRS:
        path = Path(folder) / filename
        dfs.append(pl.read_csv(path))

    return pl.concat(dfs, how="vertical_relaxed")

In [80]:
routes = load_gtfs("routes.txt")
trips = load_gtfs("trips.txt")
stops = load_gtfs("stops.txt")
stop_times = load_gtfs("stop_times.txt")
shapes = load_gtfs("shapes.txt")
calendar = load_gtfs("calendar.txt")
calendar_dates=load_gtfs("calendar_dates.txt")

In [81]:
routes = routes.unique(
    subset=["route_id"]
)

trips = trips.unique(
    subset=["trip_id"]
)

stop_times = stop_times.unique(
    subset=[
        "trip_id",
        "stop_sequence",
        "stop_id"
    ]
)

stops = stops.unique(
    subset=["stop_id"]
)

shapes = shapes.unique(
    subset=[
        "shape_id",
        "shape_pt_sequence"
    ]
)

calendar = calendar.unique(
    subset=["service_id"]
)
calendar_dates = calendar_dates.unique(
    subset=["service_id", "date", "exception_type"]
)

In [82]:
TARGET_ROUTES = [
    "M1",
    "M2",
    "M4",
    "M15",
    "M101"
]

In [83]:
routes = routes.filter(
    pl.col("route_short_name").is_in(TARGET_ROUTES)
)

In [84]:
routes

route_id,agency_id,route_short_name,route_long_name,route_desc,route_type,route_color,route_text_color
str,str,str,str,str,i64,str,str
"""M15""","""MTA NYCT""","""M15""","""East Harlem - South Ferry""","""via 1st Av / 2nd Av""",3,"""006CB7""","""FFFFFF"""
"""M4""","""MTA NYCT""","""M4""","""The Cloisters - 32 St""","""via 5th Av / Madison Av / Broa…",3,"""00AEEF""","""FFFFFF"""
"""M1""","""MTA NYCT""","""M1""","""Harlem - East Village""","""via 5th Av / Madison Av""",3,"""EE352E""","""FFFFFF"""
"""M101""","""MTA NYCT""","""M101""","""East Village - Fort George""","""via Third Av / Lexington Av / …",3,"""FAA61A""","""FFFFFF"""
"""M2""","""MTA NYCT""","""M2""","""Washington Heights - East Vill…","""via 5th Av / Madison Av / AC P…",3,"""B933AD""","""FFFFFF"""


In [85]:
trips = trips.join(
    routes.select("route_id"),
    on="route_id",
    how="inner"
)

In [86]:
stop_times = stop_times.join(
    trips.select("trip_id"),
    on="trip_id",
    how="inner"
)

In [87]:
stops = stops.join(
    stop_times.select("stop_id").unique(),
    on="stop_id",
    how="inner"
)

In [88]:
shapes = shapes.join(
    trips.select("shape_id").unique(),
    on="shape_id",
    how="inner"
)

In [89]:
stops = stops.with_columns([
    pl.col("stop_lat")
      .str.strip_chars()
      .cast(pl.Float64),

    pl.col("stop_lon")
      .str.strip_chars()
      .cast(pl.Float64),
])

In [90]:
print(routes.schema)
print(trips.schema)
print(stop_times.schema)
print(stops.schema)
print(shapes.schema)

Schema([('route_id', String), ('agency_id', String), ('route_short_name', String), ('route_long_name', String), ('route_desc', String), ('route_type', Int64), ('route_color', String), ('route_text_color', String)])
Schema([('route_id', String), ('service_id', String), ('trip_id', String), ('trip_headsign', String), ('direction_id', Int64), ('block_id', Int64), ('shape_id', String)])
Schema([('trip_id', String), ('arrival_time', String), ('departure_time', String), ('stop_id', Int64), ('stop_sequence', Int64), ('pickup_type', Int64), ('drop_off_type', Int64), ('timepoint', Int64)])
Schema([('stop_id', Int64), ('stop_name', String), ('stop_desc', String), ('stop_lat', Float64), ('stop_lon', Float64), ('zone_id', String), ('stop_url', String), ('location_type', Int64), ('parent_station', String)])
Schema([('shape_id', String), ('shape_pt_lat', Float64), ('shape_pt_lon', Float64), ('shape_pt_sequence', Int64)])


In [91]:
OUTPUT = Path("raw/processed_gtfs")
OUTPUT.mkdir(exist_ok=True)

routes.write_parquet(OUTPUT / "routes.parquet")
trips.write_parquet(OUTPUT / "trips.parquet")
stops.write_parquet(OUTPUT / "stops.parquet")
stop_times.write_parquet(OUTPUT / "stop_times.parquet")
shapes.write_parquet(OUTPUT / "shapes.parquet")
calendar.write_parquet(OUTPUT / "calendar.parquet")
calendar_dates.write_parquet(OUTPUT/"calendar_dates.parquet")

In [92]:
assert trips["trip_id"].is_unique().all()

In [93]:
assert routes["route_id"].is_unique().all()

In [94]:
assert stops["stop_id"].is_unique().all()

In [95]:
duplicates = (
    stop_times
    .group_by(["trip_id", "stop_sequence"])
    .len()
    .filter(pl.col("len") > 1)
)

print(duplicates.height)

0


In [96]:
duplicates = (
    shapes
    .group_by(["shape_id", "shape_pt_sequence"])
    .len()
    .filter(pl.col("len") > 1)
)

print(duplicates.height)

0


In [97]:
stop_times["trip_id"].unique()

trip_id
str
"""OF_B6-Weekday-138500_M1_138"""
"""OF_H6-Weekday-041900_M1_111"""
"""MV_A6-Weekday-SDon-060000_M4_4…"
"""OH_B6-Sunday-096100_M15_236"""
"""OH_B6-Weekday-SDon-122000_M101…"
…
"""MV_B6-Weekday-092700_M4_433"""
"""OH_B6-Sunday-097400_M101_37"""
"""MV_B6-Weekday-095700_M3_321"""
